In [6]:
import json
import pandas as pd
from collections import Counter
import re


# 1. Загрузка и предварительная обработка
def parse_year(date_str):
    """Извлекает год из строки вида '29 Apr, 2015' или '2020'"""
    if not date_str:
        return None
    match = re.search(r'\b(19|20)\d{2}\b', date_str)
    return int(match.group()) if match else None


# Загружаем данные
with open('steam_top_1000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Берём первые 1000 записей
data = data[:1000]

# Преобразуем в DataFrame
df = pd.DataFrame(data)

# Добавляем колонку года
df['year'] = df['release_date'].apply(parse_year)

# Убираем записи без года
df = df.dropna(subset=['year']).copy()
df['year'] = df['year'].astype(int)

# Убираем записи без тегов
df = df[df['tags'].apply(lambda x: isinstance(x, list) and len(x) > 0)]


# 2. Анализ по годам
results = {}

for year, group in df.groupby('year'):
    # Пропускаем годы с менее чем 2 играми
    if len(group) < 2:
        continue

    # Собираем все теги за год
    all_tags = [tag for tags in group['tags'] for tag in tags]
    if not all_tags:
        continue

    tag_counts = Counter(all_tags)

    # Пропускаем годы, где ни один тег не повторяется (все теги уникальны)
    if max(tag_counts.values()) == 1:
        continue

    # Три самых популярных тега
    most_common = [tag for tag, _ in tag_counts.most_common(3)]
    most_common_set = set(most_common)

    # Остальные теги (не в топ-3)
    other_tags = {tag: cnt for tag, cnt in tag_counts.items() if tag not in most_common_set}

    # Определяем наименее популярный тег (если возможно)
    least_common_tag_display = "— (нет данных)"
    actual_least_tag = None

    if other_tags:
        unique_counts = set(other_tags.values())
        if len(unique_counts) == 1:
            # Все "остальные" теги одинаково редки — нельзя выделить один
            least_common_tag_display = "— (все остальные теги уникальны и равнозначны)"
        else:
            min_count = min(other_tags.values())
            actual_least_tag = next(tag for tag, cnt in other_tags.items() if cnt == min_count)
            least_common_tag_display = actual_least_tag
    else:
        least_common_tag_display = "— (нет тегов вне топ-3)"

    # Собираем игры по тегам (только для реальных тегов)
    target_tags = most_common.copy()
    games_by_tag = {}
    for tag in target_tags:
        games = group[group['tags'].apply(lambda tags: tag in tags)]['title'].tolist()
        games_by_tag[tag] = games

    # Добавляем игры для actual_least_tag, если он есть
    if actual_least_tag is not None:
        games = group[group['tags'].apply(lambda tags: actual_least_tag in tags)]['title'].tolist()
        games_by_tag[actual_least_tag] = games

    results[year] = {
        'most_popular_tags': most_common,
        'least_popular_tag_display': least_common_tag_display,
        'actual_least_tag': actual_least_tag,
        'games_by_tag': games_by_tag
    }


# 3. Вывод результатов
output_lines = []

if not results:
    output_lines.append("Нет данных для анализа (недостаточно игр или отсутствуют повторяющиеся теги).")
else:
    for year in sorted(results.keys()):
        info = results[year]
        output_lines.append(f"\n=== Год: {year} ===")
        output_lines.append(f"Три самых популярных тега: {', '.join(info['most_popular_tags'])}")
        output_lines.append(f"Самый непопулярный тег: {info['least_popular_tag_display']}")

        output_lines.append("\nИгры по тегам:")
        for tag in info['most_popular_tags']:
            games = info['games_by_tag'][tag]
            output_lines.append(f"  Тег '{tag}':")
            for game in games[:5]:  # ← по умолчанию 5 игр
                output_lines.append(f"    - {game}")
            if len(games) > 5:
                output_lines.append(f"    ... и ещё {len(games) - 5} игр")

        # Выводим блок для наименее популярного, только если он определён
        if info['actual_least_tag'] is not None:
            tag = info['actual_least_tag']
            games = info['games_by_tag'][tag]
            output_lines.append(f"  Тег '{tag}' (наименее популярный):")
            for game in games[:5]:
                output_lines.append(f"    - {game}")
            if len(games) > 5:
                output_lines.append(f"    ... и ещё {len(games) - 5} игр")

# Сохраняем в файл
with open('analysis_results.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(output_lines))

# Выводим в консоль
print('\n'.join(output_lines))
print("\nРезультаты также сохранены в файл: analysis_results.txt")


=== Год: 2004 ===
Три самых популярных тега: Singleplayer, Classic, Atmospheric
Самый непопулярный тег: — (все остальные теги уникальны и равнозначны)

Игры по тегам:
  Тег 'Singleplayer':
    - Call of Duty: United Offensive
    - Neighbours from Hell Compilation
  Тег 'Classic':
    - Call of Duty: United Offensive
    - Neighbours from Hell Compilation
  Тег 'Atmospheric':
    - Call of Duty: United Offensive
    - Neighbours from Hell Compilation

=== Год: 2006 ===
Три самых популярных тега: Multiplayer, Action, Singleplayer
Самый непопулярный тег: Racing

Игры по тегам:
  Тег 'Multiplayer':
    - FlatOut 2
    - Urban Rivals
    - Microsoft Flight Simulator X: Steam Edition
  Тег 'Action':
    - FlatOut 2
    - Urban Rivals
    - Microsoft Flight Simulator X: Steam Edition
  Тег 'Singleplayer':
    - FlatOut 2
    - Microsoft Flight Simulator X: Steam Edition
  Тег 'Racing' (наименее популярный):
    - FlatOut 2

=== Год: 2007 ===
Три самых популярных тега: Singleplayer, Puzzle, 